# Task 15: Enterprise LLM Gateway with Dynamic Rate-Limiting and Fallback Governance

## Architecture
Token bucket rate limiting and provider router with automated failover.


In [1]:
import time

# Token bucket rate limiter and provider router with automated failover
class TokenBucketRateLimiter:
    def __init__(self, capacity=3, refill_rate=1.0):
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.tokens = capacity
        self.last_refill = time.time()

    def allow_request(self):
        now = time.time()
        self.tokens = min(self.capacity, self.tokens + (now - self.last_refill) * self.refill_rate)
        self.last_refill = now
        
        if self.tokens >= 1.0:
            self.tokens -= 1.0
            return True, 200
        return False, 429

class LLMGatewayRouter:
    def __init__(self):
        self.primary_healthy = True
        
    def route_query(self, prompt: str):
        if self.primary_healthy:
            return "Primary Provider Llama-3-8B (Status 200 OK)"
        return "Secondary Provider Mistral-7B (Fallback Active 200 OK)"


In [2]:
# Simulate API traffic burst through Enterprise Gateway with primary provider failure fallback
limiter = TokenBucketRateLimiter(capacity=3)
router = LLMGatewayRouter()

requests = [
    ("Client A", "Summarize article"),
    ("Client B", "Translate text"),
    ("Client C", "Generate code"),
    ("Client D", "Analyze sentiment"), # Should trigger HTTP 429
    ("Client E", "Answer question")    # Should trigger fallback route
]

print(f"{'Client':<10} | {'Status':<10} | {'Gateway Routing Output':<45}")
print("-" * 70)

for i, (client, msg) in enumerate(requests):
    if i == 4:
        router.primary_healthy = False # Simulate primary outage before request 5
    allowed, status = limiter.allow_request()
    if allowed:
        res = router.route_query(msg)
        print(f"{client:<10} | {status:<10} | {res:<45}")
    else:
        print(f"{client:<10} | {status:<10} | {'Blocked: Token bucket rate limit exceeded':<45}")


Client     | Status     | Gateway Routing Output                       
----------------------------------------------------------------------
Client A   | 200        | Primary Provider Llama-3-8B (Status 200 OK)  
Client B   | 200        | Primary Provider Llama-3-8B (Status 200 OK)  
Client C   | 200        | Primary Provider Llama-3-8B (Status 200 OK)  
Client D   | 429        | Blocked: Token bucket rate limit exceeded    
Client E   | 429        | Blocked: Token bucket rate limit exceeded    
